##### Import the Libraries Required:
I have everything under a Conda Environment

In [ ]:
#!pip install -q librosa numpy 
import librosa
import numpy as np
import os

## Audio Feature Extraction Notes
#### Sampling Rate (sr)

• Defines how many audio samples are captured per second.

• Higher sampling rates → higher quality, but also more memory and slower processing.

• 16,000 Hz is the standard for speech, alarms, and environmental sounds.

• 22,050–44,100 Hz is used for music or high-fidelity recordings.

• 8,000 Hz is used for telephony or low-power devices.

Tip: 16 kHz gives the best balance between clarity and efficiency for most machine-learning audio tasks.

#### Number of Mels (n_mels)
• Controls the vertical resolution (pitch detail) in the mel spectrogram.

• More mel bands → finer frequency detail, but larger data size.

Tip: Use 64 mel bands for alarm/siren detection. It is clear enough and efficient for training.

In [ ]:
sample_rate = 16000
mel_bands = 64

def extract_mel_features(audio_file):
    y, sr = librosa.load(audio_file, sr=sample_rate)
    features = []
    
    window_samples = 1 * sample_rate         # 1 second
    hop_samples = int(0.5 * sample_rate)     # 0.5 second hop

    # Helper to process and save
    def process_segment(seg):
        
        #Calculate how loud the segment is
        rms = np.sqrt(np.mean(seg**2))

        if rms < 0.005: 
            return
        
        # This Ensure segment is exactly window_samples long
        if len(seg) < window_samples:
            pad_width = window_samples - len(seg)
            seg = np.pad(seg, (0, pad_width), mode='constant')
        
        # Standard Mel
        mel = librosa.feature.melspectrogram(y=seg, sr=sample_rate,  n_mels=mel_bands)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        # Shape should be exactly (64, 32)
        # If it's off by 1 pixel due to rounding, we crop or pad
        target_width = 32 # 16000 / 512 hop roughly equals 32 frames
        if mel_db.shape[1] > target_width:
            mel_db = mel_db[:, :target_width]
        elif mel_db.shape[1] < target_width:
            mel_db = np.pad(mel_db, ((0,0), (0, target_width - mel_db.shape[1])))
            
        features.append(mel_db)

        # Noise Augmentation
        noise = np.random.normal(0, 0.005, len(seg))
        y_noise = seg + noise
        mel_noise = librosa.feature.melspectrogram(y=y_noise, sr=sample_rate,  n_mels=mel_bands)
        mel_noise_db = librosa.power_to_db(mel_noise, ref=np.max)
        
        if mel_noise_db.shape[1] > target_width:
            mel_noise_db = mel_noise_db[:, :target_width]
        elif mel_noise_db.shape[1] < target_width:
            mel_noise_db = np.pad(mel_noise_db, ((0,0), (0, target_width - mel_noise_db.shape[1])))
            
        features.append(mel_noise_db)

    # Main Loop
    for start in range(0, len(y), hop_samples):
        segment = y[start:start + window_samples]
        
        # If segment is too short (less than 0.1s), skip it to avoid empty data
        if len(segment) < (0.1 * sample_rate):
            continue
            
        process_segment(segment)
        
    return features

In [6]:
# Create Folder if it doesn't exist
os.makedirs("features", exist_ok=True)

# Throw every single audio file and create a mel spectrogram then save it into a feature folder
for class_name in ['Noises/appliance', 'Noises/carbon', 'Noises/smoke', 'Noises/siren']:
    directory = f'./{class_name}'
    for audio_file in os.listdir(directory):
        if audio_file.endswith(('.wav')):
            print(f"Extracting {audio_file}...")
            features = extract_mel_features(f"{directory}/{audio_file}")


            for i, feature in enumerate(features):
                safe_class_name = class_name.replace('/', '_')
                # Remove file extension properly for both .wav
                base_name = audio_file.rsplit('.', 1)[0]
                np.save(f'features/{base_name}_{i}.npy', feature)

Extracting appliance_003.wav...
Extracting appliance_002.wav...
Extracting appliance_001.wav...
Extracting appliance_005.wav...
Extracting appliance_001.wav...
Extracting appliance_005.wav...
Extracting appliance_004.wav...
Extracting appliance_010.wav...
Extracting appliance_004.wav...
Extracting appliance_010.wav...
Extracting appliance_006.wav...
Extracting appliance_006.wav...
Extracting appliance_007.wav...
Extracting appliance_009.wav...
Extracting appliance_008.wav...
Extracting appliance_007.wav...
Extracting appliance_009.wav...
Extracting appliance_008.wav...
Extracting carbon_045.wav...
Extracting carbon_044.wav...
Extracting carbon_045.wav...
Extracting carbon_044.wav...
Extracting carbon_046.wav...
Extracting carbon_046.wav...
Extracting carbon_043.wav...
Extracting carbon_043.wav...
Extracting carbon_042.wav...
Extracting carbon_042.wav...
Extracting carbon_040.wav...
Extracting carbon_040.wav...
Extracting carbon_041.wav...
Extracting carbon_041.wav...
Extracting carbon_